## cài đặt thư viện


In [ ]:
# Nếu môi trường chưa có thư viện, bỏ comment dòng dưới để cài đặt
# %pip install -q pandas numpy scikit-learn matplotlib seaborn joblib

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, make_scorer
)
from sklearn.base import clone

RANDOM_STATE = 42
TARGET = "Attack_type"

def find_data_file(filename: str) -> Path:
    # Tìm file theo vòng lên thư mục cha, không phụ thuộc notebook được chạy từ đâu.
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / filename,
        cwd / "splitted data" / filename,
        cwd / ".." / "splitted data" / filename,
        cwd / ".." / ".." / "splitted data" / filename,
    ]
    # Search project data folders from the notebook directory upwards.
    for parent in (cwd, *cwd.parents):
        candidates.extend([
            parent / "splitted data" / filename,
            parent / "preprocessing" / "data" / filename,
        ])

    for path in candidates:
        path = path.resolve()
        if path.is_file():
            return path

    searched = "\n".join(f"  - {path.resolve()}" for path in candidates)
    raise FileNotFoundError(f"Không tìm thấy {filename}. Đã kiểm tra:\n{searched}")

TRAIN_PATH = Path(r"C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\data\splitted data\train.csv")
VALID_PATH = Path(r"C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\data\splitted data\validation.csv")
RESULT_DIR = Path(r"C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\result\exps\exp4\E06_RF")
RESULT_DIR.mkdir(parents=True, exist_ok=True)


## đọc dữ liệu và kiểm tra cấu trúc


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
valid_df = pd.read_csv(VALID_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", valid_df.shape)
print("\\nCác cột train:")
print(train_df.columns.tolist())

if TARGET not in train_df.columns or TARGET not in valid_df.columns:
    raise ValueError(f"Không tìm thấy cột nhãn '{TARGET}' trong cả hai tập dữ liệu.")

# Chỉ tách X/y, không loại bỏ hay biến đổi đặc trưng.
X_train = train_df.drop(columns=[TARGET]).copy()
y_train = train_df[TARGET].copy()
X_valid = valid_df.drop(columns=[TARGET]).copy()
y_valid = valid_df[TARGET].copy()

missing_cols = [c for c in X_train.columns if c not in X_valid.columns]
extra_cols = [c for c in X_valid.columns if c not in X_train.columns]
if missing_cols or extra_cols:
    raise ValueError(
        f"Cột không khớp giữa train và validation. "
        f"Thiếu ở validation: {missing_cols}; dư ở validation: {extra_cols}"
    )
X_valid = X_valid[X_train.columns]

# Không tự động điền giá trị thiếu vì yêu cầu không xử lý dữ liệu.
if X_train.isna().any().any() or X_valid.isna().any().any():
    raise ValueError("Dữ liệu có giá trị thiếu. Notebook không tự impute; cần xử lý nguồn dữ liệu trước.")

print("\\nSố lớp train:", y_train.nunique())
print("Phân bố lớp train:")
display(y_train.value_counts())


## xây dựng pipeline E06


In [ ]:
numeric_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

# Cột số giữ nguyên (passthrough); chỉ mã hóa cột phân loại.
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ],
    remainder="passthrough"
)

rf_e06 = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    class_weight="balanced_subsample",
    n_jobs=-1
)

e06_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", rf_e06)
])

print("Số cột số:", len(numeric_cols))
print("Số cột phân loại:", len(categorical_cols))
print("Các cột phân loại:", categorical_cols)
print(e06_model)


## Cross-validation 3-fold trên tập train


In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "macro_f1": make_scorer(f1_score, average="macro", zero_division=0),
    "weighted_f1": make_scorer(f1_score, average="weighted", zero_division=0),
    "macro_precision": make_scorer(precision_score, average="macro", zero_division=0),
    "macro_recall": make_scorer(recall_score, average="macro", zero_division=0),
}

cv_results = cross_validate(
    e06_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1,
    error_score="raise"
)

cv_summary = pd.DataFrame({
    "metric": list(scoring.keys()),
    "mean": [cv_results[f"test_{m}"].mean() for m in scoring],
    "std": [cv_results[f"test_{m}"].std(ddof=1) for m in scoring],
    "fold_1": [cv_results[f"test_{m}"][0] for m in scoring],
    "fold_2": [cv_results[f"test_{m}"][1] for m in scoring],
    "fold_3": [cv_results[f"test_{m}"][2] for m in scoring],
})
display(cv_summary.style.format({c: "{:.4f}" for c in cv_summary.columns if c != "metric"}))


## Huấn luyện trên toàn bộ train và đánh giá validation


In [ ]:
# Fit lại mô hình trên toàn bộ tập train
e06_model.fit(X_train, y_train)

# Dự đoán validation
y_pred = e06_model.predict(X_valid)

validation_scores = {
    "accuracy": accuracy_score(y_valid, y_pred),
    "macro_f1": f1_score(y_valid, y_pred, average="macro", zero_division=0),
    "weighted_f1": f1_score(y_valid, y_pred, average="weighted", zero_division=0),
    "macro_precision": precision_score(y_valid, y_pred, average="macro", zero_division=0),
    "macro_recall": recall_score(y_valid, y_pred, average="macro", zero_division=0),
}

validation_summary = pd.DataFrame(
    [{"model": "E06_RF_balanced_subsample", **validation_scores}]
)
display(validation_summary.style.format({
    k: "{:.4f}" for k in validation_scores
}))

print(classification_report(y_valid, y_pred, zero_division=0))


## Confusion matrix


In [ ]:
labels = e06_model.named_steps["classifier"].classes_
cm = confusion_matrix(y_valid, y_pred, labels=labels)

plt.figure(figsize=(max(9, len(labels) * 0.7), max(7, len(labels) * 0.6)))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=labels, yticklabels=labels
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("E06 – Confusion Matrix (Validation)")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(RESULT_DIR / "E06_validation_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


## Lưu bảng metric và kết quả đánh giá


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

fold_metrics = pd.DataFrame({
    "fold": np.arange(1, len(cv_results["test_accuracy"]) + 1),
    **{metric: cv_results[f"test_{metric}"] for metric in scoring},
})
fold_metrics.to_csv(RESULT_DIR / "E06_cv_folds.csv", index=False, encoding="utf-8-sig")
cv_summary.to_csv(RESULT_DIR / "E06_cv_summary.csv", index=False, encoding="utf-8-sig")
validation_summary.to_csv(RESULT_DIR / "E06_validation_metrics.csv", index=False, encoding="utf-8-sig")

precision, recall, f1, support = precision_recall_fscore_support(
    y_valid, y_pred, labels=labels, zero_division=0
)
per_class = pd.DataFrame({
    "class": labels,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "support": support,
})
per_class.to_csv(RESULT_DIR / "E06_validation_per_class.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(cm, index=pd.Index(labels, name="Actual"), columns=labels).to_csv(
    RESULT_DIR / "E06_validation_confusion_matrix.csv", encoding="utf-8-sig"
)
pd.DataFrame({"y_true": y_valid.to_numpy(), "y_pred": y_pred}).to_csv(
    RESULT_DIR / "E06_validation_predictions.csv", index=False, encoding="utf-8-sig"
)
print("Saved E06 fold, validation, class, confusion-matrix tables and plot to:", RESULT_DIR)
